# TAMP-OS — Resolution Comparison Notebook

Run a single image through the pipeline at multiple parameter values to compare how **resolution**, **relief height**, and **blur** affect the output.

Each run produces a clearly named file (e.g. `elephant_resolution_64.stl`, `elephant_resolution_256.stl`) so you can open and compare them side by side in MeshLab or PrusaSlicer.

---
### Step 1 — Install dependencies (run once)

In [9]:
!pip install numpy pillow scipy numpy-stl matplotlib

### Step 2 — Imports & pipeline functions
*(self-contained — no external files needed)*

In [10]:
import os, sys, struct, json, zipfile
from pathlib import Path
import numpy as np
from PIL import Image
from scipy.ndimage import gaussian_filter
from stl import mesh
import matplotlib.pyplot as plt

# ── Height map ────────────────────────────────────────────────────────────────
def image_to_heightmap(image_path, output_size=(512,512), invert=False,
                       blur_sigma=1.0, contrast_percentile=(2.0,98.0),
                       preserve_aspect=True, flip=True):
    img = Image.open(image_path).convert("L")
    orig_w, orig_h = img.size
    if preserve_aspect and orig_w != orig_h:
        target_w = output_size[0]
        target_h = round(target_w * orig_h / orig_w)
        actual_size = (target_w, target_h)
        print(f"  [i] {orig_w}x{orig_h} -> {target_w}x{target_h}")
    else:
        actual_size = output_size
    img = img.resize(actual_size, Image.LANCZOS)
    arr = np.array(img, dtype=np.float32)
    lo = np.percentile(arr, contrast_percentile[0])
    hi = np.percentile(arr, contrast_percentile[1])
    arr = np.clip((arr - lo) / (hi - lo + 1e-8), 0.0, 1.0)
    if blur_sigma > 0:
        arr = gaussian_filter(arr, sigma=blur_sigma)
    if invert:
        arr = 1.0 - arr
    if flip:
        arr = np.flipud(arr)
    return arr

# ── STL mesh ──────────────────────────────────────────────────────────────────
def heightmap_to_stl(heightmap, output_path, base_thickness_mm=1.0,
                     max_relief_mm=3.0, physical_size_mm=(100.0,100.0)):
    rows, cols = heightmap.shape
    dx = physical_size_mm[0] / (cols - 1)
    dy = physical_size_mm[1] / (rows - 1)
    hm_ratio = cols / rows
    print_ratio = physical_size_mm[0] / physical_size_mm[1]
    if abs(hm_ratio - print_ratio) > 0.05:
        print(f"  [WARNING] Aspect ratio mismatch — consider size_y={round(physical_size_mm[0]/hm_ratio,1)}")
    z_top = base_thickness_mm + heightmap * max_relief_mm
    num_tris = (rows-1)*(cols-1)*4 + 2*((rows-1)+(cols-1))*2
    litho_mesh = mesh.Mesh(np.zeros(num_tris, dtype=mesh.Mesh.dtype))
    tri_idx = 0
    def add_tri(v0, v1, v2):
        nonlocal tri_idx
        litho_mesh.vectors[tri_idx] = [v0, v1, v2]
        tri_idx += 1
    for r in range(rows-1):
        for c in range(cols-1):
            x0,y0 = c*dx, r*dy
            x1 = (c+1)*dx
            x2,y2 = c*dx, (r+1)*dy
            x3,y3 = (c+1)*dx, (r+1)*dy
            z00,z10,z01,z11 = z_top[r,c],z_top[r,c+1],z_top[r+1,c],z_top[r+1,c+1]
            add_tri([x0,y0,z00],[x1,y0,z10],[x2,y2,z01])
            add_tri([x1,y0,z10],[x3,y3,z11],[x2,y2,z01])
            add_tri([x0,y0,0],[x2,y2,0],[x1,y0,0])
            add_tri([x1,y0,0],[x2,y2,0],[x3,y3,0])
    xmax,ymax = (cols-1)*dx,(rows-1)*dy
    for c in range(cols-1):
        x0,x1 = c*dx,(c+1)*dx
        z0,z1 = z_top[0,c],z_top[0,c+1]
        add_tri([x0,0,0],[x1,0,0],[x0,0,z0])
        add_tri([x1,0,0],[x1,0,z1],[x0,0,z0])
    for c in range(cols-1):
        x0,x1 = c*dx,(c+1)*dx
        z0,z1 = z_top[-1,c],z_top[-1,c+1]
        add_tri([x0,ymax,0],[x0,ymax,z0],[x1,ymax,0])
        add_tri([x1,ymax,0],[x0,ymax,z0],[x1,ymax,z1])
    for r in range(rows-1):
        y0,y1 = r*dy,(r+1)*dy
        z0,z1 = z_top[r,0],z_top[r+1,0]
        add_tri([0,y0,0],[0,y0,z0],[0,y1,0])
        add_tri([0,y1,0],[0,y0,z0],[0,y1,z1])
    for r in range(rows-1):
        y0,y1 = r*dy,(r+1)*dy
        z0,z1 = z_top[r,-1],z_top[r+1,-1]
        add_tri([xmax,y0,0],[xmax,y1,0],[xmax,y0,z0])
        add_tri([xmax,y1,0],[xmax,y1,z1],[xmax,y0,z0])
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    litho_mesh.save(str(output_path))
    print(f"  [OK] STL -> {output_path}")
    return Path(output_path)

# ── Format converters ─────────────────────────────────────────────────────────
def stl_to_3mf(stl_path):
    from stl import mesh as stl_mesh
    m = stl_mesh.Mesh.from_file(str(stl_path))
    vertices = []; vert_map = {}; indices = []
    for tri in m.vectors:
        face = []
        for v in tri:
            key = tuple(float(x) for x in v)
            if key not in vert_map:
                vert_map[key] = len(vertices)
                vertices.append(key)
            face.append(vert_map[key])
        indices.append(face)
    vx = " ".join(f'<vertex x="{v[0]:.4f}" y="{v[1]:.4f}" z="{v[2]:.4f}"/>' for v in vertices)
    tx = " ".join(f'<triangle v1="{t[0]}" v2="{t[1]}" v3="{t[2]}"/>' for t in indices)
    ns = "http://schemas.microsoft.com/3dmanufacturing/core/2015/02"
    xml = (f'<?xml version="1.0" encoding="UTF-8"?>'
           f'<model unit="millimeter" xmlns="{ns}">'
           f'<resources><object id="1" type="model"><mesh>'
           f'<vertices>{vx}</vertices><triangles>{tx}</triangles>'
           f'</mesh></object></resources><build><item objectid="1"/></build></model>')
    out = Path(str(stl_path).replace(".stl", ".3mf"))
    ct_ns = "http://schemas.openxmlformats.org/package/2006/content-types"
    rel_ns = "http://schemas.openxmlformats.org/package/2006/relationships"
    rel_type = "http://schemas.microsoft.com/3dmanufacturing/2013/01/3dmodel"
    with zipfile.ZipFile(str(out), "w", zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("3D/3dmodel.model", xml)
        zf.writestr("[Content_Types].xml",
            f'<?xml version="1.0"?><Types xmlns="{ct_ns}">'
            f'<Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/>'
            f'<Default Extension="model" ContentType="application/vnd.ms-package.3dmanufacturing-3dmodel+xml"/>'
            f'</Types>')
        zf.writestr("_rels/.rels",
            f'<?xml version="1.0"?><Relationships xmlns="{rel_ns}">'
            f'<Relationship Type="{rel_type}" Target="/3D/3dmodel.model" Id="r1"/>'
            f'</Relationships>')
    print(f"  [OK] 3MF -> {out}")
    return out

def stl_to_glb(stl_path):
    from stl import mesh as stl_mesh
    m = stl_mesh.Mesh.from_file(str(stl_path))
    verts = m.vectors.reshape(-1, 3).astype(np.float32)
    indices = np.arange(len(verts), dtype=np.uint32)
    def pad4(b): return b + b'\x00' * ((4 - len(b) % 4) % 4)
    ib = pad4(indices.tobytes())
    vb = pad4(verts.tobytes())
    bd = ib + vb
    gltf = {
        "asset": {"version": "2.0", "generator": "TAMP-OS"}, "scene": 0,
        "scenes": [{"nodes": [0]}], "nodes": [{"mesh": 0}],
        "meshes": [{"primitives": [{"attributes": {"POSITION": 1}, "indices": 0}]}],
        "accessors": [
            {"bufferView": 0, "componentType": 5125, "count": len(indices), "type": "SCALAR"},
            {"bufferView": 1, "componentType": 5126, "count": len(verts), "type": "VEC3",
             "min": verts.min(0).tolist(), "max": verts.max(0).tolist()}],
        "bufferViews": [
            {"buffer": 0, "byteOffset": 0, "byteLength": len(ib), "target": 34963},
            {"buffer": 0, "byteOffset": len(ib), "byteLength": len(vb), "target": 34962}],
        "buffers": [{"byteLength": len(bd)}]
    }
    jb = pad4(json.dumps(gltf, separators=(",", ":")).encode())
    def chunk(d, t): return struct.pack("<II", len(d), t) + d
    out = Path(str(stl_path).replace(".stl", ".glb"))
    with open(str(out), "wb") as f:
        f.write(struct.pack("<III", 0x46546C67, 2, 12 + 8 + len(jb) + 8 + len(bd))
                + chunk(jb, 0x4E4F534A) + chunk(bd, 0x004E4942))
    print(f"  [OK] GLB -> {out}")
    return out

print("All functions loaded OK.")


All functions loaded OK.


---
## ⚙️ Parameters — edit this cell

This is the only cell you need to edit before running.

In [15]:
# ── Image ─────────────────────────────────────────────────────────────────────
# Full path to your image, or just the filename if it's in the same folder
IMAGE_PATH = r"C:\Users\vazquez\Downloads\LithographsFolder\Asian_elephant_trunk_HeidelbergZoo_1.jpg"

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = "./comparison_output"   # folder where files will be saved

# ── Output formats ────────────────────────────────────────────────────────────
SAVE_STL = True    # .STL  — universal, works in any slicer
SAVE_3MF = False   # .3MF  — best for PrusaSlicer
SAVE_GLB = False   # .GLB  — best for web viewers

# ── Fixed parameters (same across ALL runs) ───────────────────────────────────
SIZE_X       = 100.0   # print width in mm
SIZE_Y       = 100.0   # print height in mm (100 = square, or match aspect ratio)
BASE_MM      = 1.0     # base thickness in mm
FLIP         = True    # flip vertically to match image orientation (recommended)
INVERT       = False   # invert relief (dark areas raised) — for some TEM images

# ── Which parameter to sweep ──────────────────────────────────────────────────
# Pick ONE: "resolution", "relief_height", or "blur"
SWEEP_PARAM = "blur"

# ── Values to test for each parameter ────────────────────────────────────────
RESOLUTION_VALUES    = [64, 128, 256, 512]   # pixels — lower = smaller file, less detail
RELIEF_HEIGHT_VALUES = [1.0, 2.0, 3.0, 5.0] # mm    — lower = subtle, higher = pronounced
BLUR_VALUES          = [0.5, 1.0, 1.5, 2.0] # sigma — lower = sharp, higher = smooth

# ── Default values for parameters that are NOT being swept ───────────────────
DEFAULT_RESOLUTION    = 256
DEFAULT_RELIEF_HEIGHT = 3.0
DEFAULT_BLUR          = 1.2

# ─────────────────────────────────────────────────────────────────────────────
# Summary (auto-generated — do not edit below this line)
# ─────────────────────────────────────────────────────────────────────────────
SWEEP_VALUES = {
    "resolution":    RESOLUTION_VALUES,
    "relief_height": RELIEF_HEIGHT_VALUES,
    "blur":          BLUR_VALUES,
}
DEFAULTS = {
    "resolution":    DEFAULT_RESOLUTION,
    "relief_height": DEFAULT_RELIEF_HEIGHT,
    "blur":          DEFAULT_BLUR,
}
vals = SWEEP_VALUES[SWEEP_PARAM]

param_labels = {
    "resolution":    "Resolution (px)",
    "relief_height": "Relief height (mm)",
    "blur":          "Blur (sigma)",
}

print("=" * 55)
print("  TAMP-OS Resolution Comparison — Run Plan")
print("=" * 55)
print(f"  Image:          {IMAGE_PATH}")
print(f"  Output folder:  {OUTPUT_DIR}")
print(f"  Formats:        STL={SAVE_STL}  3MF={SAVE_3MF}  GLB={SAVE_GLB}")
print(f"  Fixed:          size={SIZE_X}x{SIZE_Y} mm, base={BASE_MM} mm")
print(f"  Sweeping:       {param_labels[SWEEP_PARAM]}")
print(f"  Values:         {vals}")
print(f"  Defaults:       resolution={DEFAULT_RESOLUTION}, relief={DEFAULT_RELIEF_HEIGHT}, blur={DEFAULT_BLUR}")
print(f"  Total runs:     {len(vals)}")
print("=" * 55)
for v in vals:
    int_v = int(v) if SWEEP_PARAM == "resolution" else v
    stem = Path(IMAGE_PATH).stem
    print(f"  • {param_labels[SWEEP_PARAM]} = {int_v:>6}  ->  {stem}_{SWEEP_PARAM}_{int_v}.stl")


  TAMP-OS Resolution Comparison — Run Plan
  Image:          C:\Users\vazquez\Downloads\LithographsFolder\Asian_elephant_trunk_HeidelbergZoo_1.jpg
  Output folder:  ./comparison_output
  Formats:        STL=True  3MF=False  GLB=False
  Fixed:          size=100.0x100.0 mm, base=1.0 mm
  Sweeping:       Blur (sigma)
  Values:         [0.5, 1.0, 1.5, 2.0]
  Defaults:       resolution=256, relief=3.0, blur=1.2
  Total runs:     4
  • Blur (sigma) =    0.5  ->  Asian_elephant_trunk_HeidelbergZoo_1_blur_0.5.stl
  • Blur (sigma) =    1.0  ->  Asian_elephant_trunk_HeidelbergZoo_1_blur_1.0.stl
  • Blur (sigma) =    1.5  ->  Asian_elephant_trunk_HeidelbergZoo_1_blur_1.5.stl
  • Blur (sigma) =    2.0  ->  Asian_elephant_trunk_HeidelbergZoo_1_blur_2.0.stl


---
## ▶ Run the comparison
Run this cell to generate all files.

In [16]:
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
stem = Path(IMAGE_PATH).stem
results = []

for val in vals:
    p = dict(DEFAULTS)
    p[SWEEP_PARAM] = val
    int_v = int(val) if SWEEP_PARAM == "resolution" else val
    label = f"{stem}_{SWEEP_PARAM}_{int_v}"
    print(f"\n[{vals.index(val)+1}/{len(vals)}] {param_labels[SWEEP_PARAM]} = {int_v}")

    hm = image_to_heightmap(
        IMAGE_PATH,
        output_size=(p["resolution"], p["resolution"]),
        blur_sigma=p["blur"],
        flip=FLIP,
        invert=INVERT,
        preserve_aspect=True,
    )

    stl_path = Path(OUTPUT_DIR) / f"{label}.stl"
    heightmap_to_stl(
        hm, stl_path,
        base_thickness_mm=BASE_MM,
        max_relief_mm=p["relief_height"],
        physical_size_mm=(SIZE_X, SIZE_Y),
    )

    if SAVE_3MF: stl_to_3mf(stl_path)
    if SAVE_GLB:  stl_to_glb(stl_path)
    if not SAVE_STL: stl_path.unlink(missing_ok=True)

    size_mb = os.path.getsize(stl_path) / 1e6 if SAVE_STL and stl_path.exists() else 0
    results.append({"val": int_v, "label": label, "hm": hm, "size_mb": size_mb})

print(f"\nDone! {len(vals)} files saved to {Path(OUTPUT_DIR).resolve()}")



[1/4] Blur (sigma) = 0.5
  [i] 3717x3712 -> 256x256
  [OK] STL -> comparison_output\Asian_elephant_trunk_HeidelbergZoo_1_blur_0.5.stl

[2/4] Blur (sigma) = 1.0
  [i] 3717x3712 -> 256x256
  [OK] STL -> comparison_output\Asian_elephant_trunk_HeidelbergZoo_1_blur_1.0.stl

[3/4] Blur (sigma) = 1.5
  [i] 3717x3712 -> 256x256
  [OK] STL -> comparison_output\Asian_elephant_trunk_HeidelbergZoo_1_blur_1.5.stl

[4/4] Blur (sigma) = 2.0
  [i] 3717x3712 -> 256x256
  [OK] STL -> comparison_output\Asian_elephant_trunk_HeidelbergZoo_1_blur_2.0.stl

Done! 4 files saved to C:\Users\vazquez\Documents\TAMP-OS\comparison_output


---
## 📊 Visual comparison — height maps side by side

In [ ]:
n = len(results)
fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
fig.patch.set_facecolor('#0f0f0f')
if n == 1: axes = [axes]

for ax, r in zip(axes, results):
    ax.set_facecolor('#0f0f0f')
    im = ax.imshow(r["hm"], cmap='inferno', aspect='auto', vmin=0, vmax=1)
    ax.set_title(
        f"{param_labels[SWEEP_PARAM]} = {r['val']}\nshape: {r['hm'].shape}",
        color='white', fontsize=11, pad=8)
    ax.axis('off')

fig.suptitle(
    f"Height map comparison — sweeping {param_labels[SWEEP_PARAM]}",
    color='white', fontsize=14, y=1.02)

plt.tight_layout(pad=1.5)
preview_path = f"{OUTPUT_DIR}/comparison_{SWEEP_PARAM}.png"
plt.savefig(preview_path, dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print(f"Preview saved to {preview_path}")


---
## 📁 File summary

In [ ]:
print(f"Files saved to: {Path(OUTPUT_DIR).resolve()}\n")
for r in results:
    formats_saved = []
    for ext in [".stl", ".3mf", ".glb"]:
        p = Path(OUTPUT_DIR) / f"{r['label']}{ext}"
        if p.exists():
            formats_saved.append(f"{ext.upper()} ({os.path.getsize(p)/1e6:.1f} MB)")
    sizes = ",  ".join(formats_saved) if formats_saved else "no file found"
    print(f"  {param_labels[SWEEP_PARAM]} = {r['val']:>6}  ->  {sizes}")
